# Lesson 09: Testing, CI, Docker

**Goal:** write a parametrized pytest test and read what the CI pipeline runs.

**Before you start**
- Theory: [lesson 09](../../docs/lessons/09_testing_ci_docker.md)
- Python concepts: pytest, `@pytest.mark.parametrize`, YAML
- Files used: `src/docrag/shared/text.py`, `.github/workflows/ci.yml`. Offline, no API key, no cost.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import tempfile
from pathlib import Path

import pytest
import yaml

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)

## Step 1: A parametrized test

`@pytest.mark.parametrize(("raw", "expected"), [...])` runs one test function once per row, and each row is reported as its own test, like JUnit's `@ParameterizedTest` or a table-driven test in Go/Rust. Write a test module (as a string) for `normalize()` and `tokenize()` from `docrag.shared.text`. The next cell writes it to a temp folder and runs pytest on it.

What the functions do: `normalize` lowercases, unifies curly quotes, rejoins `infor-\nmation` and squashes spaces. `tokenize` drops stopwords and keeps numbers like `1,500` whole.

In [ ]:
# TODO: replace the placeholder with a parametrized test of normalize() (at least 2 rows)
#       and one test that tokenize("We sold 1,500 units") contains "1,500".
TEST_CODE = """
from docrag.shared.text import normalize, tokenize

def test_placeholder():
    assert False, "write your tests here"
"""

In [ ]:
tmp = Path(tempfile.mkdtemp())
(tmp / "test_lesson09.py").write_text(TEST_CODE, encoding="utf-8")
# In-process pytest run on just this file (no cache folder, rootdir = the temp folder).
exit_code = pytest.main(
    [str(tmp), "-q", "-p", "no:cacheprovider", "--rootdir", str(tmp)]
)
assert "parametrize" in TEST_CODE, "Use @pytest.mark.parametrize"
assert exit_code == 0, "Some tests failed: read the pytest output above"
print("OK")

## Step 2: What does CI run?

`.github/workflows/ci.yml` is plain YAML. Load it with `yaml.safe_load` (never `yaml.load`, which can construct arbitrary objects) and list the job names.

In [ ]:
# TODO: ci = yaml.safe_load(...) of ROOT / ".github" / "workflows" / "ci.yml"; jobs = list(ci["jobs"])
ci = {}
jobs = []

In [ ]:
assert {"lint-test", "e2e", "docker"} <= set(jobs), (
    f"Expected the three CI jobs, got {jobs}"
)
for name in jobs:
    steps = [s.get("name", s.get("uses", "?")) for s in ci["jobs"][name]["steps"]]
    print(f"{name}: {steps}")
print("e2e services:", list(ci["jobs"]["e2e"]["services"]))
# YAML 1.1 gotcha: the key `on:` is parsed as the boolean True.
print(
    "trigger key parsed as:",
    [k for k in ci if k not in ("name", "jobs", "concurrency", "env", "permissions")],
)
print("OK")

## Observation question

Why does the E2E test not need a GPU (or an API key)?

<details>
<summary>Answer</summary>

The E2E job runs the real pipeline against a real Qdrant (started as a CI `service` container), but with `configs/e2e_fake.yaml`: the fake embedder hashes words, the fake reranker counts word overlap, and the fake LLM cites a real quote. So it tests the wiring, the Qdrant queries, the API and Verify end to end, deterministically and for free. What it can't test is model *quality*; that's what the benchmark runs are for.
</details>

## The real code

- Unit tests: [`tests/unit`](../../tests/unit)
- E2E test: [`tests/e2e/test_e2e.py`](../../tests/e2e/test_e2e.py)
- CI: [`.github/workflows/ci.yml`](../../.github/workflows/ci.yml)
- Text utils: [`src/docrag/shared/text.py`](../../src/docrag/shared/text.py)
- Lesson: [`docs/lessons/09_testing_ci_docker.md`](../../docs/lessons/09_testing_ci_docker.md)